In [ ]:
import math


N_python = 20000      # eligible Python posts after filtering
N_java = 5300         # eligible Java posts
N_total = N_python + N_java

n_per_lang = 2000     # sampled per language
n_total = 4000        # total sample size

p = 0.5               # worst-case for MoE and MDD
z = 1.96              # 95% CI
z_power = 0.84        # 80% power (z for beta=0.20)

def finite_population_correction(N, n):
    return math.sqrt((N - n) / (N - 1))

def margin_of_error(N, n, p=0.5):
    base = z * math.sqrt((p * (1 - p)) / n)
    fpc = finite_population_correction(N, n)
    return base * fpc

def mdd_two_sample(N, n, p=0.5):
    """
    MDD for two-sample proportions (n per group),
    then scaled by finite-population correction.
    """
    base = (z + z_power) * math.sqrt(2 * p * (1 - p) / n)
    fpc = finite_population_correction(N, n)
    return base * fpc

moe_python = margin_of_error(N_python, n_per_lang)
moe_java = margin_of_error(N_java, n_per_lang)
moe_total = margin_of_error(N_total, n_total)

mdd_python = mdd_two_sample(N_python, n_per_lang)
mdd_java = mdd_two_sample(N_java, n_per_lang)


print("=== Margin of Error (95% CI) ===")
print(f"Python: {moe_python*100:.2f}%")
print(f"Java:   {moe_java*100:.2f}%")
print(f"Total:  {moe_total*100:.2f}%")

print("\n=== Minimum Detectable Difference (α=0.05, power=0.80) ===")
print(f"Python: {mdd_python*100:.2f}%")
print(f"Java:   {mdd_java*100:.2f}%")


=== Margin of Error (95% CI) ===
Python: 2.08%
Java:   1.73%
Total:  1.42%

=== Minimum Detectable Difference (α=0.05, power=0.80) ===
Python: 4.20%
Java:   3.49%


In [ ]:
import pandas as pd
from scipy.stats import ks_2samp

def clean_numeric(df, cols):
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    return df

cols = ["score", "code_length"]

pop_py = clean_numeric(pd.read_csv("/content/Superset Python.csv"), cols)
samp_py = clean_numeric(pd.read_csv("/content/Collective Python for representative test.csv"), cols)

pop_ja = clean_numeric(pd.read_csv("/content/Superset Java.csv"), cols)
samp_ja = clean_numeric(pd.read_csv("/content/Collective Java for representative test.csv"), cols)

def compare(pop, samp, label):
    print(f"\n=== {label} ===")
    for col in ["score", "code_length"]:
        pop_col = pop[col].dropna()
        samp_col = samp[col].dropna()

        pop_mean = pop_col.mean()
        samp_mean = samp_col.mean()
        delta = abs(pop_mean - samp_mean) / pop_mean * 100
        ks_p = ks_2samp(pop_col, samp_col).pvalue

        print(f"{col}:")
        print(f"   Population mean = {pop_mean:.2f}")
        print(f"   Sample mean     = {samp_mean:.2f}")
        print(f"   Δ (%)           = {delta:.2f}%")
        print(f"   KS-test p-value = {ks_p:.4f}")

compare(pop_py, samp_py, "PYTHON")
compare(pop_ja, samp_ja, "JAVA")



=== PYTHON ===
score:
   Population mean = 1.89
   Sample mean     = 6.45
   Δ (%)           = 242.05%
   KS-test p-value = 0.0000
code_length:
   Population mean = 34.36
   Sample mean     = 36.35
   Δ (%)           = 5.79%
   KS-test p-value = 0.0246

=== JAVA ===
score:
   Population mean = 2.03
   Sample mean     = 2.28
   Δ (%)           = 12.17%
   KS-test p-value = 0.0000
code_length:
   Population mean = 41.71
   Sample mean     = 42.28
   Δ (%)           = 1.37%
   KS-test p-value = 0.0380
